# Week 31, Day 2: Container protocols

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w031d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Week:
    DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    def __getitem__(self, i):
        return self.DAYS[i]

w = Week()
print(w[0], w[-1], w[1:3])
print("Fri" in w, [d[0] for d in w])

In [ ]:
class Show:
    def __getitem__(self, key):
        return key

s = Show()
print(s[3], s["Lagos"], s[2, 5])
print(s[1:4], s[::-1], s[1:4, 0])

In [ ]:
class Playlist:
    def __init__(self, name, songs=()):
        self.name = name
        self._songs = list(songs)

    def __len__(self):
        return len(self._songs)

    def __getitem__(self, key):
        if isinstance(key, slice):
            return type(self)(self.name, self._songs[key])     # a new Playlist
        return self._songs[key]                                # one song

    def __repr__(self):
        return f"Playlist({self.name!r}, {self._songs!r})"

p = Playlist("Road trip", ["Ye", "Essence", "Calm Down", "Peru"])
print(p[0], p[-1], p[1:3])
try:
    p[10]
except IndexError as err:
    print("IndexError:", err)

In [ ]:
class Scores:
    """Exam scores by name. Every score is a whole number from 0 to 100."""

    def __init__(self):
        self._data = {}

    def __setitem__(self, name, score):
        if not isinstance(score, int) or isinstance(score, bool) or not 0 <= score <= 100:
            raise ValueError(f"bad score for {name}: {score!r}")
        self._data[name] = score

    def __getitem__(self, name):
        return self._data[name]

    def __delitem__(self, name):
        del self._data[name]

    def __len__(self):
        return len(self._data)

s = Scores()
s["Ada"] = 72
s["Tunde"] = 88
s["Ada"] += 5                   # reads with __getitem__, writes with __setitem__
del s["Tunde"]
print(s["Ada"], len(s))
for bad in [101, "90", True]:
    try:
        s["Bo"] = bad
    except ValueError as err:
        print("ValueError:", err)

In [ ]:
class TimeSlot:
    """Minutes past midnight, from start up to (not including) end."""

    def __init__(self, start, end):
        self.start = start
        self.end = end

    def __contains__(self, minute):
        return self.start <= minute < self.end

lunch = TimeSlot(12 * 60, 13 * 60 + 30)
print(12 * 60 + 45 in lunch, 13 * 60 + 30 in lunch, 9 * 60 not in lunch)

In [ ]:
class Team:
    def __init__(self, name, players):
        self.name = name
        self._players = list(players)

    def __iter__(self):
        return iter(self._players)

    def __len__(self):
        return len(self._players)

t = Team("Rivers United", ["Ojo", "Ama", "Ude"])
for player in t:
    print(player)
print(list(t), sorted(t), ", ".join(t), len(t))
first, *rest = t
print(first, rest)

In [ ]:
class LowerDict:
    def __init__(self, data=None):
        self._data = {}
        for key, value in (data or {}).items():
            self[key] = value                     # through __setitem__

    def __setitem__(self, key, value):
        self._data[key.lower()] = value

    def __getitem__(self, key):
        return self._data[key.lower()]

    def __delitem__(self, key):
        del self._data[key.lower()]

    def __contains__(self, key):
        return key.lower() in self._data

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def get(self, key, default=None):
        return self[key] if key in self else default

d = LowerDict({"Lagos": 15})
d["LEEDS"] = 0.8
d["leeds"] += 0.1
print(d["lagos"], round(d["Leeds"], 1), "KANO" in d, len(d), sorted(d))
del d["LAGOS"]
print(list(d), d.get("Lagos", "gone"))

> **Key idea.** Container syntax calls special methods: `x[k]` calls `__getitem__` (with an int, a key, a tuple for `x[a, b]`, or a `slice` object for `x[i:j]`), `x[k] = v` calls `__setitem__`, `del x[k]` calls `__delitem__`, `v in x` calls `__contains__`, and `for v in x` calls `__iter__`, which should return `iter(...)` of the data. Sequences raise `IndexError` and usually return their own type for slices; mappings raise `KeyError` and iterate over their keys. Check values in `__setitem__`, and route every way in through it. With `__iter__` and `__len__` your class works with `list`, `sorted`, `sum`, `max`, unpacking and comprehensions.

### Exercise: A deck that works with random

Give `Deck` a `__len__` and a `__getitem__` (positions and slices both handed to the list of cards; a slice may simply return a list). That's all `random.choice`, `reversed` and `in` need. Then write `top(deck, n)`, the last `n` cards of the deck as a list, using a slice.

In [ ]:
import random

class Deck:
    def __init__(self):
        self._cards = [r + s for s in "♠♥♦♣" for r in ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]]

    ___

def top(deck, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deck():\n    \"a deck\"\n    import random as rnd\n    D = ns[\"Deck\"]\n    d = D()\n    assert len(d) == 52 and d[0] == \"A\u2660\" and d[-1] == \"K\u2663\" and d[13] == \"A\u2665\"\n    assert d[0:3] == [\"A\u2660\", \"2\u2660\", \"3\u2660\"] and \"Q\u2666\" in d and \"Z\u2660\" not in d\n    assert rnd.choice(d) in d._cards and next(reversed(d)) == \"K\u2663\"\n    assert top(d, 2) == [\"Q\u2663\", \"K\u2663\"] and top(d, 0) == []\n    try:\n        d[52]\n        assert False, \"d[52] should raise IndexError\"\n    except IndexError:\n        pass")

### Exercise: A span of numbers

Write `Span(start, stop)` for the whole numbers from `start` up to, but not including, `stop` (if `stop` is below `start`, the span is empty). Give it `__len__`, `__contains__` (True only for whole numbers in the span, worked out without looping), `__iter__` (the numbers in order) and `repr` `Span(3, 7)`.

In [ ]:
class Span:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_span():\n    \"spans\"\n    S = ns[\"Span\"]\n    s = S(3, 7)\n    assert len(s) == 4 and list(s) == [3, 4, 5, 6] and sum(s) == 18 and repr(s) == \"Span(3, 7)\"\n    assert 3 in s and 6 in s and 7 not in s and 2 not in s and 4.5 not in s and \"4\" not in s\n    e = S(5, 2)\n    assert len(e) == 0 and list(e) == [] and 3 not in e and not e\n    big = S(0, 10**9)\n    assert 10**9 - 1 in big and 10**9 not in big and len(big) == 10**9")

### Exercise: A gradebook

Write `Gradebook`, a mapping from student names to marks, by composition:

- `gb[name] = mark`: `name` must be a non-empty string and `mark` a whole number from 0 to 100 (not a `bool`); otherwise `ValueError`.
- `gb[name]` gives the mark (`KeyError` if missing); `del gb[name]` removes it.
- `name in gb`, `len(gb)`, and iteration over the names in **alphabetical** order.
- `Gradebook(marks)` can take a starting dictionary, which goes through the same checks.
- `average()` gives the mean mark rounded to 1 place, or `None` if empty.

In [ ]:
class Gradebook:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basics():\n    \"marks\"\n    G = ns[\"Gradebook\"]\n    gb = G({\"Tunde\": 64, \"Ada\": 72})\n    gb[\"Bisi\"] = 90\n    gb[\"Ada\"] += 3\n    assert gb[\"Ada\"] == 75 and len(gb) == 3 and \"Bisi\" in gb and \"Zara\" not in gb\n    assert list(gb) == [\"Ada\", \"Bisi\", \"Tunde\"] and gb.average() == 76.3\n    del gb[\"Bisi\"]\n    assert list(gb) == [\"Ada\", \"Tunde\"] and G().average() is None\n    try:\n        gb[\"Zara\"]\n        assert False, \"a missing name should raise KeyError\"\n    except KeyError:\n        pass\ndef test_checks():\n    \"checks on every way in\"\n    G = ns[\"Gradebook\"]\n    gb = G()\n    for name, mark in [(\"Ada\", 101), (\"Ada\", -1), (\"Ada\", 7.5), (\"Ada\", True), (\"\", 50), (3, 50)]:\n        try:\n            gb[name] = mark\n            assert False, f\"gb[{name!r}] = {mark!r} should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        G({\"Ada\": 200})\n        assert False, \"starting marks must be checked too\"\n    except ValueError:\n        pass\n    assert len(gb) == 0")

### Exercise: A grid with two coordinates

Write `Grid(rows, cols, fill=".")`, a rectangle of single characters, reached with **two** coordinates: `g[r, c]` and `g[r, c] = "#"`. Coordinates must be whole numbers inside the grid (no negative ones); anything else raises `IndexError`. A value that isn't a single character raises `ValueError`. Also:

- `"#" in g` is True if any cell holds `"#"`;
- iterating gives the rows as strings, top to bottom;
- `str(g)` is the rows joined by `"\n"`.

In [ ]:
class Grid:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_grid():\n    \"a grid\"\n    G = ns[\"Grid\"]\n    g = G(2, 3)\n    g[0, 1] = \"#\"\n    g[1, 2] = \"@\"\n    assert g[0, 1] == \"#\" and g[1, 0] == \".\" and list(g) == [\".#.\", \"..@\"] and str(g) == \".#.\\n..@\"\n    assert \"#\" in g and \"*\" not in g and str(G(1, 2, \"~\")) == \"~~\"\ndef test_errors():\n    \"errors\"\n    G = ns[\"Grid\"]\n    g = G(2, 3)\n    for r, c in [(2, 0), (0, 3), (-1, 0), (0, -1)]:\n        for action in [\"get\", \"set\"]:\n            try:\n                if action == \"get\":\n                    g[r, c]\n                else:\n                    g[r, c] = \"#\"\n                assert False, f\"g[{r}, {c}] should raise IndexError\"\n            except IndexError:\n                pass\n    for bad in [\"\", \"##\", 5]:\n        try:\n            g[0, 0] = bad\n            assert False, f\"setting {bad!r} should raise ValueError\"\n        except ValueError:\n            pass\n    assert str(g) == \"...\\n...\"")

### Bug hunt: None for a missing key

A missing code should print `unknown: XX`, but this prints `None`. The problem is in `__getitem__`. Fix it so the output is `Ikeja` and then `unknown: XX`.

In [ ]:
class Stops:
    def __init__(self, pairs):
        self._pairs = list(pairs)

    def __getitem__(self, code):
        for k, name in self._pairs:
            if k == code:
                return name

stops = Stops([("IK", "Ikeja"), ("YB", "Yaba")])
for code in ["IK", "XX"]:
    try:
        print(stops[code])
    except KeyError:
        print("unknown:", code)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"missing keys raise KeyError\"\n    expect_output(\"Ikeja\\nunknown: XX\")")

### Bug hunt: Not an iterator

Looping over a `Shelf` crashes with `TypeError: iter() returned non-iterator of type 'list'`. Fix it so this prints `Aké, Kindred`.

In [ ]:
class Shelf:
    def __init__(self, titles):
        self._titles = list(titles)

    def __iter__(self):
        return self._titles

print(", ".join(Shelf(["Aké", "Kindred"])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"joined\"\n    expect_output(\"Ak\u00e9, Kindred\")")

### Bug hunt: A bad price slips in

`Prices` should never hold a negative price, but `Prices({"rice": -100})` is accepted, and this prints `-100` instead of `refused`. Fix `__init__` so starting prices are checked too, without copying the check.

In [ ]:
class Prices:
    def __init__(self, start):
        self._data = dict(start)

    def __setitem__(self, item, pence):
        if pence < 0:
            raise ValueError(f"negative price for {item}")
        self._data[item] = pence

    def __getitem__(self, item):
        return self._data[item]

try:
    p = Prices({"rice": -100})
    print(p["rice"])
except ValueError:
    print("refused")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"refused\"\n    expect_output(\"refused\")\n    p = ns[\"Prices\"]({\"oil\": 900})\n    assert p[\"oil\"] == 900")

### Puzzle: A bag of marbles

Arrange the lines so that a `Bag` supports `len`, `in` and `for`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Echo:
    def __getitem__(self, key):
        return key

class Box:
    def __init__(self):
        self.items = [5, 6, 7]
    def __iter__(self):
        return iter(self.items)
    def __contains__(self, x):
        return x == 99

e = Echo()
b = Box()

p = ___   # e[1, 2]
q = ___   # type(e[1:2]).__name__
r = ___   # 6 in b
s = ___   # 99 in b
t = ___   # sum(b)
u = ___   # sorted(b, reverse=True)[0]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (1, 2), \"p\"\n    assert q == \"slice\", \"q\"\n    assert (r, s, t, u) == (False, True, 18, 7)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a day's timetable

Write `Timetable`, a mapping from times of day to events.

- Times are strings like `"9:05"` or `"09:05"`: hours 0 to 23, then a colon, then exactly two digits of minutes 0 to 59. They're stored in the form `"09:05"`, so `"9:05"` and `"09:05"` are the same key. A bad time raises `ValueError` when setting, getting or deleting; `in` simply returns False for a bad time.
- `tt[time] = event` (an event is a non-empty string, otherwise `ValueError`), `tt[time]` (`KeyError` if there's nothing then), `del tt[time]`, `time in tt`, `len(tt)`.
- Iterating gives the times in order, earliest first.
- **Slicing by time**: `tt["09:00":"12:00"]` gives a **new** `Timetable` with the events from 9:00 up to, but not including, 12:00. A missing start means from midnight, a missing end means to the end of the day. A step raises `ValueError`.
- `Timetable(events)` can take a starting dictionary, which goes through the same checks.
- `repr` gives `Timetable({'09:00': 'Assembly', '10:30': 'Maths'})`, in time order.

In [ ]:
import re

class Timetable:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def T(*a):\n    return ns[\"Timetable\"](*a)\ndef test_basics():\n    \"times and events\"\n    tt = T({\"9:00\": \"Assembly\", \"13:15\": \"Lunch\"})\n    tt[\"10:30\"] = \"Maths\"\n    tt[\"8:05\"] = \"Register\"\n    assert tt[\"09:00\"] == \"Assembly\" and tt[\"9:00\"] == \"Assembly\" and len(tt) == 4\n    assert list(tt) == [\"08:05\", \"09:00\", \"10:30\", \"13:15\"]\n    assert \"8:05\" in tt and \"08:06\" not in tt and \"25:00\" not in tt and \"nonsense\" not in tt\n    del tt[\"8:05\"]\n    assert list(tt) == [\"09:00\", \"10:30\", \"13:15\"]\n    assert repr(T({\"10:30\": \"Maths\", \"9:00\": \"Assembly\"})) == \"Timetable({'09:00': 'Assembly', '10:30': 'Maths'})\"\n    try:\n        tt[\"11:00\"]\n        assert False, \"a free time should raise KeyError\"\n    except KeyError:\n        pass\ndef test_bad():\n    \"bad times and events\"\n    tt = T()\n    for bad in [\"24:00\", \"9:60\", \"9:5\", \"09-00\", \"\", \"123:00\", \"9:005\"]:\n        try:\n            tt[bad] = \"x\"\n            assert False, f\"time {bad!r} should raise ValueError\"\n        except ValueError:\n            pass\n    for event in [\"\", 5, None]:\n        try:\n            tt[\"10:00\"] = event\n            assert False, f\"event {event!r} should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        T({\"30:00\": \"x\"})\n        assert False, \"starting events must be checked\"\n    except ValueError:\n        pass\n    assert len(tt) == 0\ndef test_slices():\n    \"slicing by time\"\n    tt = T({\"8:05\": \"Register\", \"9:00\": \"Assembly\", \"10:30\": \"Maths\", \"12:00\": \"Lunch\", \"15:30\": \"Home\"})\n    morning = tt[\"9:00\":\"12:00\"]\n    assert type(morning).__name__ == \"Timetable\" and list(morning) == [\"09:00\", \"10:30\"]\n    assert list(tt[:\"9:00\"]) == [\"08:05\"] and list(tt[\"12:00\":]) == [\"12:00\", \"15:30\"] and len(tt[:]) == 5\n    morning[\"11:00\"] = \"Break\"\n    assert \"11:00\" not in tt, \"a slice should be a new timetable\"\n    try:\n        tt[\"9:00\":\"12:00\":2]\n        assert False, \"a step should raise ValueError\"\n    except ValueError:\n        pass")

## Warm-up

`Week` has a single special method, yet indexing, negative indexes, slices, `in` and a `for` loop all work. Indexing and slicing call `__getitem__` directly; `in` and `for` fall back on it, trying `w[0]`, `w[1]`, `w[2]` and so on until an `IndexError` says there's no more. Today you'll see each piece of container syntax, the method it calls, and why it's usually better to write the specific methods than to rely on fallbacks.

## Getting items

`x[key]` calls `type(x).__getitem__(x, key)`. What `key` is depends on what was written in the brackets:

- `s[3]` passes the integer `3`; `s["Lagos"]` passes the string.
- `s[2, 5]` passes the **tuple** `(2, 5)`: the commas make a tuple, as everywhere in Python. That's how grids and tables take two coordinates.
- `s[1:4]` passes a **slice object**, `slice(1, 4, None)`, with `start`, `stop` and `step` attributes. Python doesn't slice anything for you; your method decides what a slice means.

A **sequence** (reached by position, like a list) usually stores a list and hands the key to it, which gets negative indexes, slices and the `IndexError` for free. For a slice, a well-behaved sequence returns a new object of its own type, as lists and strings do:

A **mapping** (reached by key, like a dictionary) usually stores a dictionary. A missing key must raise `KeyError`, not return `None`: that's what callers (and `except KeyError`) rely on.

## Setting and deleting items

`x[key] = value` calls `__setitem__(self, key, value)` and `del x[key]` calls `__delitem__(self, key)`. Leave them out and your container is read-only, which is often right. When you do write `__setitem__`, it's the natural place to check values:

Notice `s["Ada"] += 5`: Python reads the old value with `__getitem__`, adds, and writes back with `__setitem__`, so the new value is checked too.

Every way in should go through the check. If `__init__` takes starting data, add it with `self[name] = score` rather than straight into `self._data`, or bad values can slip in at the start. That was exactly the `dict` subclass trouble in Week 30: its constructor skipped the overridden `__setitem__`. With composition, you control every way in.

## Membership

`x in container` calls `__contains__(self, x)`, which should return True or False. Without it, Python falls back on `__iter__`, and then on `__getitem__` with 0, 1, 2 and so on, checking each item with `==`. The fallbacks work, but writing `__contains__` lets you:

- make it **fast**: checking a dictionary's keys or a set takes the same short time however big it is (Week 17), while the fallback checks items one by one;
- give `in` the **right meaning**: for a mapping, `in` should check the keys, like a dictionary; for a range of dates, it can check whether a date lies inside without listing every day.

`not in` comes for free: Python inverts `__contains__`.

## Iteration

`for v in x` calls `iter(x)`, which calls `__iter__`. It must return an **iterator**, an object that hands out the items one at a time. Phase 4 explains iterators and how to write them; for now, the simple way is to return an iterator over data you already hold:

Once `__iter__` exists, a lot comes with it: `list()`, `sorted()`, `sum()`, `max()`, `", ".join()`, unpacking, comprehensions, and `in` (as a fallback). That's duck typing again (Week 30, Day 4): these functions only need something they can iterate.

Two conventions: a **mapping**'s `__iter__` gives its **keys**, like a dictionary's; and `__iter__` should return `iter(...)` of something, not the list itself. A list is *iterable* but isn't an *iterator*, and returning it raises `TypeError: iter() returned non-iterator`. You'll fix that in the practice.

## Sequences and mappings

Here's what each kind of container usually defines:

| | Sequence (by position) | Mapping (by key) |
|---|---|---|
| `__len__` | number of items | number of keys |
| `__getitem__` | int index or slice; `IndexError` if out of range | key; `KeyError` if missing |
| `__setitem__`, `__delitem__` | if it can change | if it can change |
| `__contains__` | is this value an item? | is this a key? |
| `__iter__` | the items, in order | the keys |

Here's Week 30's case-insensitive dictionary, finished properly by composition, with every piece of syntax going through the lower-casing:

Writing all of this by hand is fine for small classes. Week 32 shows `collections.abc`, whose base classes fill in the extra methods (`get`, `keys`, `index`, `count` and more) once you've written the core ones.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a class from Weeks 28 to 30 that holds a collection (a basket, a playlist, a league) and give it `__len__`, `__iter__` and `__contains__`. Notice how much simpler the code that uses it becomes. Tomorrow: rich comparisons, so `<`, `sorted` and `max` work on your objects.